# Financial Single Agent：单条测试

这个 Notebook 用最小配置验证完整链路：

`输入问题 → 规划 1 个章节 → Tavily 搜索 1 次 → 总结 → Markdown 报告`

默认关闭反思，因此正常情况下只产生 **1 次 Tavily 搜索请求**。运行前请确认项目根目录已有 `.env`。

## 1. 安装项目

首次使用当前 Kernel 时运行下一格。已经执行过 `pip install -e .` 可以跳过。

In [1]:
%pip install -e .

Obtaining file:///Users/luoyuwen/Desktop/projects/financial_single_agent
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Using cached tavily_python-0.7.26-py3-none-any.whl.metadata (12 kB)
Using cached tavily_python-0.7.26-py3-none-any.whl (21 kB)
  Building editable for financial-single-agent (pyproject.toml) ... done
  Created wheel for financial-single-agent: filename=financial_single_agent-0.1.0-0.editable-py3-none-any.whl size=3937 sha256=15d0fa70c87af75111e5da3e1d152f08dae38f682b3d7c42360d273280b67be5
  Stored in directory: /private/var/folders/cp/bpqz82f9367fdyzwdrf0qq400000gn/T/pip-ephem-wheel-cache-v49byto7/wheels/af/9a/85/4b1ac02191c775c3dc0548e3f12be2efc4a2b9ad7209bb7469
Successfully built financial-single-agent
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [financial-single-agent]
Note: you may need to restart th

如果上一步是首次安装，并且下一格仍提示找不到模块，请重启一次 Notebook Kernel。

In [1]:
from financial_single_agent.agent import FinancialResearchAgent
from financial_single_agent.utils.config import Settings
from IPython.display import Markdown, display

## 2. 检查配置

只检查密钥是否存在，不显示密钥内容。

In [2]:
base_config = Settings()

print("LLM 模型：", base_config.QUERY_ENGINE_MODEL_NAME)
print("LLM Base URL：", base_config.QUERY_ENGINE_BASE_URL or "OpenAI 默认地址")
print("LLM API Key 已配置：", bool(base_config.QUERY_ENGINE_API_KEY))
print("Tavily API Key 已配置：", bool(base_config.TAVILY_API_KEY))

LLM 模型： deepseek-chat
LLM Base URL： https://api.deepseek.com
LLM API Key 已配置： True
Tavily API Key 已配置： True


## 3. 设置最小测试参数

- `MAX_PARAGRAPHS=1`：只规划一个章节
- `MAX_REFLECTIONS=0`：不做补充搜索
- `MAX_SEARCH_RESULTS=5`：最多取五条搜索结果
- `save_report=False`：本次测试不写入报告文件

In [3]:
config = base_config.model_copy(
    update={
        "MAX_PARAGRAPHS": 1,
        "MAX_REFLECTIONS": 0,
        "MAX_SEARCH_RESULTS": 5,
    }
)

print("预计 Tavily 搜索次数：", config.MAX_PARAGRAPHS * (1 + config.MAX_REFLECTIONS))

预计 Tavily 搜索次数： 1


## 4. 输入一个金融事件并运行

In [4]:
query = "中国近期货币政策变化对银行股和债券市场可能产生哪些影响？"

agent = FinancialResearchAgent(config)
report = agent.research(query, save_report=False)

display(Markdown(report))

数据截止时间：2026-07-17T16:03+08:00

# 中国近期货币政策变化对银行股与债券市场的影响分析

## 背景与传导机制

截至2026年7月，中国货币政策呈现结构性宽松特征。2026年1月，央行下调多项结构性货币政策工具利率，被视为年内首次结构性“降息”，中国社会科学院金融研究所研究员彭兴韵在接受时代周报专访时指出，此举释放出持续降低实体经济融资成本的明确信号，并预计年内法定存款准备金率或将下调0.5个百分点左右（[来源](https://www.time-weekly.com/post/326937)）。同年1月的央行工作会议提出“促进社会综合融资成本低位运行”，而非此前“下降”的表述，东方金诚首席宏观分析师王青分析认为，这反映了央行对银行负债端利率比价约束的关注（[来源](https://www.jiemian.com/article/13853754.html)）。中邮证券研究所于2025年12月发布的展望报告则指出，鉴于央行已通过买断式逆回购与MLF等高额滚续投放中长期流动性，2026年降准的空间或有限（[来源](https://pdf.dfcfw.com/pdf/H3_AP202512031793174425_1.pdf?1764778319000.pdf=)）。

传导至**银行股**层面：降息将直接压缩净息差。中邮证券于2025年12月的报告测算，2026年政策利率仍存在约20BP的可调降空间，虽然存款利率等负债端成本的调降有助于缓冲降息对盈利的冲击，但银行净息差仍面临压力（[来源](https://pdf.dfcfw.com/pdf/H3_AP202512031793174425_1.pdf?1764778319000.pdf=)）。另一方面，彭兴韵指出，下调结构性工具利率有助于降低银行付息成本、稳定净息差（[来源](https://www.time-weekly.com/post/326937)）。中国央行《2026年第一季度货币政策执行报告》数据显示，2026年一季度银行间市场资金利率明显下行，隔夜Shibor较上年末下降5个基点至1.28%，这为银行提供了低成本的资金来源（[来源](https://jrj.sh.gov.cn/cmsres/5a/5ae23d4703d4409b8c2b2c325c40b774/b0233b6bfdbe3826e365a66f35d86f83.pdf)）。资产质量方面，彭兴韵认为，适度宽松的货币政策有助于稳定经济增长、改善企业盈利环境，从而可能对银行资产质量形成支撑（[来源](https://www.time-weekly.com/post/326937)）。综合来看，降息短期内对银行盈利构成压制，但降准（如果落地）释放的低成本资金以及存款利率下调可部分对冲息差压力，而经济复苏预期对资产质量的改善则为股价提供长期支撑。

传导至**债券市场**层面：彭兴韵分析指出，降息预期通常导致国债收益率下行，推动债券价格上升（[来源](https://www.time-weekly.com/post/326937)）。央行2026年一季度报告验证了这一趋势：2026年3月末，1年期、10年期国债收益率分别为1.22%和1.82%，较上年末分别下行12个基点和3个基点；同时，1年期与10年期国债利差走阔至60个基点，较上年末扩大9个基点，收益率曲线呈现陡峭化特征（[来源](https://jrj.sh.gov.cn/cmsres/5a/5ae23d4703d4409b8c2b2c325c40b774/b0233b6bfdbe3826e365a66f35d86f83.pdf)）。短端利率下行幅度大于长端，符合宽松政策下短券受益更直接的逻辑。然而，宽财政预期对长端利率形成扰动。中邮证券报告指出，2026年专项债与超长期国债仍将形成稳定供给，可能需要较低的利率环境来降低政府融资成本（[来源](https://pdf.dfcfw.com/pdf/H3_AP202512031793174425_1.pdf?1764778319000.pdf=)）。界面新闻报道亦称，央行预计将通过降准和买断式逆回购等工具保持流动性充裕，以配合政府债券发行（[来源](https://www.jiemian.com/article/13853754.html)）。此外，央行在2025年10月恢复了国债买卖操作，以引导国债收益率曲线平稳运行（[来源](http://www.china-cer.com.cn/guwen/2025121831152.html)）。总体而言，市场呈现短券强势、曲线陡峭化的特征，宽财政预期带来的长债供给压力可能对长端利率产生向上扰动，但央行通过多种工具进行对冲，力求维持市场稳定。

---

## 来源列表

1. **年内首次结构性“降息”落地，专家预计今年或降准0.5个百分点**，time-weekly.com，发布日期：无，URL: https://www.time-weekly.com/post/326937
2. **中国货币政策执行报告**，jrj.sh.gov.cn，发布日期：无，URL: https://jrj.sh.gov.cn/cmsres/5a/5ae23d4703d4409b8c2b2c325c40b774/b0233b6bfdbe3826e365a66f35d86f83.pdf
3. **央行定调2026年货币政策，为何提“优化供给”和“金融市场稳定”？|界面新闻**，jiemian.com，发布日期：无，URL: https://www.jiemian.com/article/13853754.html
4. **2026年货币政策将灵活高效精准发力**，china-cer.com.cn，发布日期：无，URL: http://www.china-cer.com.cn/guwen/2025121831152.html
5. **货币政策重心转移————2026 年展望系列四**，pdf.dfcfw.com，发布日期：无，URL: https://pdf.dfcfw.com/pdf/H3_AP202512031793174425_1.pdf?1764778319000.pdf=

## 风险与局限

本报告内容主要依赖公开搜索资料，所引用的公开信息、研究报告及财经新闻可能存在信息更新不及时或特定机构观点偏差。报告内涉及的债券收益率、Shibor等行情数据均为历史数据，非实时行情，存在一定时滞。报告中的所有前瞻性判断、测算及专家观点均基于截至数据截止时间（2026年7月17日）的有限信息，不排除后续市场出现与判断不符的变化。

## 免责声明

本报告内容仅供信息研究参考，不构成任何形式的投资建议或投资决策依据。作者及来源机构不对任何人因使用本报告中的信息而导致的任何直接或间接损失承担责任。投资者应依据自身判断，并充分咨询专业投资顾问后独立作出投资决策。


## 5. 检查真实调用结果

`search_history` 中每条记录对应一条搜索结果，不等于 API 请求次数。本测试只有一个章节且没有反思，所以逻辑搜索请求数固定为 1。

In [5]:
print("实际章节数：", len(agent.state.paragraphs))
print("逻辑 Tavily 搜索请求数：", len(agent.state.paragraphs) * (1 + config.MAX_REFLECTIONS))
print("返回并保存到状态的搜索结果数：", len(agent.state.source_list()))
print("数据截止时间：", agent.state.data_cutoff)

for index, source in enumerate(agent.state.source_list(), 1):
    print(f"{index}. {source.title}")
    print(f"   来源：{source.source or '未知'}")
    print(f"   发布时间：{source.published_date or '未知'}")
    print(f"   URL：{source.url}")

实际章节数： 1
逻辑 Tavily 搜索请求数： 1
返回并保存到状态的搜索结果数： 5
数据截止时间： 2026-07-17T16:03+08:00
1. 年内首次结构性“降息”落地，专家预计今年或降准0.5个百分点
   来源：time-weekly.com
   发布时间：未知
   URL：https://www.time-weekly.com/post/326937
2. 中国货币政策执行报告
   来源：jrj.sh.gov.cn
   发布时间：未知
   URL：https://jrj.sh.gov.cn/cmsres/5a/5ae23d4703d4409b8c2b2c325c40b774/b0233b6bfdbe3826e365a66f35d86f83.pdf
3. 央行定调2026年货币政策，为何提“优化供给”和“金融市场稳定”？|界面新闻
   来源：jiemian.com
   发布时间：未知
   URL：https://www.jiemian.com/article/13853754.html
4. 2026年货币政策将灵活高效精准发力
   来源：china-cer.com.cn
   发布时间：未知
   URL：http://www.china-cer.com.cn/guwen/2025121831152.html
5. 货币政策重心转移————2026 年展望系列四
   来源：pdf.dfcfw.com
   发布时间：未知
   URL：https://pdf.dfcfw.com/pdf/H3_AP202512031793174425_1.pdf?1764778319000.pdf=


## 6. 可选：保存这次报告

需要保存时取消下一格代码的注释。

In [ ]:
# report_path = agent._save_report(report)
# print("报告已保存到：", report_path)